# 结构化输出

In [1]:
# 加载环境变量
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv()


model = init_chat_model(
    model="deepseek-v4-pro",          
    # model="deepseek-chat",          
    model_provider="deepseek",
    extra_body={"thinking": {"type": "disabled"}}   # 关键：关闭思考模式
)

## 1. Pydantic

### 1.1 可选字段+默认值+限制条件

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """ A simple data model for a person."""
    name: str = Field(..., description="The person's name")
    age: int | None = Field(default=None, description="The person's age")
    occupation: str | None = Field(default="个体", description="The person's occupation")

model_with_structured_output = model.with_structured_output(Person)


structured_output = model_with_structured_output.invoke("张三")
rprint(structured_output)   # Person(name='张三', age=30, occupation='个体')

structured_output = model_with_structured_output.invoke("张三是后端工程师。")
rprint(structured_output)   # Person(name='张三', age=None, occupation='后端工程师')

structured_output = model_with_structured_output.invoke("张三是一名30岁的律师。")
rprint(structured_output)   # Person(name='张三', age=30, occupation='律师')

In [ ]:
from pydantic import BaseModel, Field

class Employee(BaseModel):
    """员工信息 — 带字段限制条件."""
    name: str = Field(..., min_length=1, max_length=50, description="员工姓名")
    age: int = Field(..., ge=0, le=150, description="年龄（0-150）")
    phone: str = Field(...,pattern=r"^1[3-9]\d{9}$",description="手机号（11位，1[3-9]开头）")
    score: float = Field(..., ge=0.0, le=100.0, description="绩效评分（0-100）")

model_with_structured_output = model.with_structured_output(Employee)

# 正常数据 — 通过校验
structured_output = model_with_structured_output.invoke("张三年龄30岁，手机号13800138000，绩效评分92.5")
rprint(structured_output)   # Employee(name='张三', age=30, phone='13800138000', score=92.5)

# 错误数据 — 不通过校验
structured_output = model_with_structured_output.invoke("张三年龄-5岁，手机号13800138000，绩效评分105")
rprint(structured_output)   # ValidationError: 2 validation errors for Employee

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ A simple data model for a movie."""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    rating: float | None  = Field(default=None, description="The movie's rating on a scale of 1 to 10")
    story: str | None = Field(default=None, description="A brief summary of the movie's plot")


model_with_structured_output = model.with_structured_output(Movie, include_raw=True)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的详细信息")
rprint(structured_output) 

In [4]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Person(BaseModel):
    """ 个人基本信息."""
    name: str = Field(..., description="The person's name")
    gender: str = Field(..., description="The person's gender")
    age: int = Field(..., description="The person's age")

class Actor(BaseModel):
    """ 演员相关信息."""
    info: Person = Field(..., description="The personal information of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")
    works: list[str] | None = Field(default=None, description="A list of other works the actor has been in")

class Movie(BaseModel):
    """ 电影相关信息."""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    rating: float  = Field(..., description="The movie's rating on a scale of 1 to 10")
    cast: list[Actor] = Field(..., description="A list of main cast members")
    story: str = Field(..., description="A brief summary of the movie's plot")


model_with_structured_output = model.with_structured_output(Movie)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的详细信息")
rprint(structured_output) 

Movie(
    title='肖申克的救赎',
    director='弗兰克·德拉邦特',
    year=1994,
    rating=9.7,
    cast=[
        Actor(
            info=Person(name='蒂姆·罗宾斯', gender='男', age=65),
            role='安迪·杜佛兰',
            works=['神秘河', '战争游戏', '百万金臂']
        ),
        Actor(
            info=Person(name='摩根·弗里曼', gender='男', age=86),
            role='艾利斯·波伊德·瑞德',
            works=['七宗罪', '百万美元宝贝', '蝙蝠侠：黑暗骑士']
        ),
        Actor(
            info=Person(name='鲍勃·冈顿', gender='男', age=78),
            role='典狱长塞缪尔·诺顿',
            works=['拆弹部队', '王牌对王牌']
        ),
        Actor(
            info=Person(name='克兰西·布朗', gender='男', age=64),
            role='狱警拜伦·哈德利',
            works=['星河战队', '魔兽世界']
        )
    ],
    story='银行家安迪·杜佛兰被错误判定谋杀妻子及其情人，被判处终身监禁，送入肖申克监狱。在狱中，他凭借自身的财务知识获得狱警和典狱长的信任，同时与囚犯瑞德成为好友。经过近20年的隐忍与坚持，安迪最终用一把小锤子挖通地道成功越狱，并揭露了典狱长的腐败。瑞德出狱后，在墨西哥与安迪重聚，迎来了自由与希望。'
)


### 1.2 枚举类型

In [ ]:
from enum import Enum
from typing import Literal  # noqa: F401


class Priority(str, Enum):
    """ 紧急程度 """
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """ 客户信息 """
    name: str = Field(..., description="客户姓名")
    email: str | None = Field(default=None, description="客户邮箱")
    phone: str | None = Field(default=None, description="客户电话")
    order_number: str | None = Field(default=None, description="订单号")
    issue: str = Field(..., description="客户问题描述")
    priority: Priority = Field(..., description="问题紧急程度")
    # priority: Literal["低", "中", "高"] = Field(..., description="问题紧急程度")

conversation = """
客服：您好，请问有什么可以帮您？
客户：我是黎明，我的订单一直没有发货，着急用，请问能帮我查一下吗？
客服：当然可以，提供一下您的订单号和联系方式
客户：订单号是123456789，电话是13800138000
客服：好的，我帮您查一下，稍后通过您留下的联系方式答复您具体情况。
"""

model_with_structured_output = model.with_structured_output(CustomerInfo)
structured_output = model_with_structured_output.invoke(conversation)
rprint(structured_output)

### 1.3 列表结构

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """ A simple data model for a person."""
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The person's age")

class PersonList(BaseModel):
    """ A simple data model for a list of persons."""
    persons: list[Person] = Field(..., description="A list of persons")

persons_text = """张三，30岁 李四，25岁 王五，40岁"""

model_with_structured_output = model.with_structured_output(PersonList)
structured_output = model_with_structured_output.invoke(persons_text)
rprint(structured_output) # PersonList(persons=[Person(name='张三', age=30), Person(name='李四', age=25), Person(name='王五', age=40)])

In [ ]:
from pydantic import BaseModel, Field

class Review(BaseModel):
    """ A simple data model for a review."""
    product: str = Field(..., description="The name of the product being reviewed")
    rating: int = Field(..., description="The rating given by the reviewer (1-5)")
    pros: list[str] = Field(..., description="A list of pros for the product")
    cons: list[str] = Field(..., description="A list of cons for the product")

review_text = """智能手表X100很棒，心率监测和运动模式经常使用，但是价格有点贵，都这个价格了也不送个充电器，8分"""

model_with_structured_output = model.with_structured_output(Review)
structured_output = model_with_structured_output.invoke(review_text)
rprint(structured_output) # Review(product='智能手表X100', rating=8, pros=['心率监测', '运动模式'], cons=['价格有点贵', '不送充电器'])

### 1.4 嵌套结构

In [ ]:
from pydantic import BaseModel, Field

class Address(BaseModel):
    """ 地址描述."""
    city: str = Field(..., description="The city")
    district: str | None = Field(default=None, description="The district of the city")

class Company(BaseModel):
    """ 公司描述."""
    name: str = Field(..., description="The name of the company")
    address: Address = Field(..., description="The address of the company")

address_text = """阿里巴巴坐落于中国浙江省杭州市西湖区"""

model_with_structured_output = model.with_structured_output(Company)
structured_output = model_with_structured_output.invoke(address_text)
rprint(structured_output) # Company(name='阿里巴巴', address=Address(city='杭州', district='西湖区'))

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Person(BaseModel):
    """ 个人基本信息."""
    name: str = Field(..., description="The person's name")
    gender: str = Field(..., description="The person's gender")
    age: int = Field(..., description="The person's age")

class Actor(BaseModel):
    """ 演员相关信息."""
    info: Person = Field(..., description="The personal information of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")
    works: list[str] | None = Field(default=None, description="A list of other works the actor has been in")

class Movie(BaseModel):
    """ 电影相关信息."""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    rating: float  = Field(..., description="The movie's rating on a scale of 1 to 10")
    cast: list[Actor] = Field(..., description="A list of main cast members")
    story: str = Field(..., description="A brief summary of the movie's plot")


model_with_structured_output = model.with_structured_output(Movie)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的详细信息")
rprint(structured_output)

## 2. TypedDict

`TypedDict` 是 Python 原生的“字典形状”类型声明。它适合轻量结构化输出：返回结果直接当 `dict` 使用。下面覆盖单对象、可选字段、枚举、列表和嵌套对象这几类常见格式。

### 2.1 单对象：可选字段 + 枚举

In [1]:
from typing import Literal  # noqa: F811
from typing_extensions import Annotated, TypedDict

class CustomerInfoDict(TypedDict):
    """客户问题信息"""
    name: Annotated[str, ..., "客户姓名"]
    phone: Annotated[str | None, ..., "客户电话，没有则为 None"]
    issue: Annotated[str, ..., "客户问题描述"]
    priority: Annotated[Literal["低", "中", "高"], ..., "问题紧急程度"]

conversation = """
客户：我是黎明，我的订单一直没有发货，着急用。
客服：请提供联系方式。
客户：电话是13800138000。
"""

model_with_structured_output = model.with_structured_output(CustomerInfoDict)
structured_output = model_with_structured_output.invoke(conversation)
rprint(structured_output)  # {'name': '黎明', 'phone': '13800138000', 'issue': '订单一直没有发货', 'priority': '高'}

NameError: name 'model' is not defined

### 2.2 列表与嵌套对象

In [ ]:
from typing_extensions import Annotated, TypedDict

class ActorDict(TypedDict):
    """演员信息"""
    name: Annotated[str, ..., "演员姓名"]
    role: Annotated[str, ..., "饰演角色"]

class MovieDict(TypedDict):
    """电影相关信息"""
    title: Annotated[str, ..., "电影名称"]
    director: Annotated[str, ..., "导演"]
    year: Annotated[int, ..., "上映年份"]
    rating: Annotated[float, ..., "评分，1-10 分"]
    cast: Annotated[list[ActorDict], ..., "主要演员列表"]

model_with_structured_output = model.with_structured_output(MovieDict)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的主要信息")
rprint(structured_output)

### 2.3 注意事项

- `TypedDict` 返回的是普通 `dict`，不是对象实例。
- `Annotated[type, ..., "字段描述"]` 可以把字段说明传给模型。
- 它主要提供结构提示和静态类型信息，运行时校验能力弱于 Pydantic。
- 如果后续业务逻辑需要严格校验，建议把返回的 `dict` 再交给 Pydantic 校验。

## 3. dataclass

`dataclass` 是 Python 标准库里的数据类，适合表达普通业务对象。它不能像 Pydantic 一样直接提供完整的运行时校验能力；在大模型结构化输出里，更稳的用法是：先用 Pydantic `TypeAdapter` 把 dataclass 转成 JSON Schema，模型返回 `dict` 后，再校验并还原成 dataclass 实例。

### 3.1 单对象：可选字段

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ContactInfo:
    """联系人信息"""
    name: str = field(metadata={"description": "联系人姓名"})
    email: str | None = field(default=None, metadata={"description": "邮箱，没有则为 None"})
    phone: str | None = field(default=None, metadata={"description": "电话，没有则为 None"})

model_with_structured_output = model.with_structured_output(ContactInfo)
response = model_with_structured_output.invoke("张三的电话是13800138000，没有邮箱。")
rprint(response)

### 3.2 列表字段与嵌套对象

In [ ]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class ActionItem:
    """会议待办事项"""
    task: str = field(metadata={"description": "需要完成的任务"})
    owner: str = field(metadata={"description": "负责人"})
    priority: Literal["低", "中", "高"] = field(metadata={"description": "优先级"})

@dataclass
class MeetingSummary:
    """会议纪要"""
    topic: str = field(metadata={"description": "会议主题"})
    decisions: list[str] = field(metadata={"description": "会议决定列表"})
    action_items: list[ActionItem] = field(metadata={"description": "待办事项列表"})

model_with_structured_output = model.with_structured_output(MeetingSummary)

text = """""
本次会议讨论上线计划，决定周五发布第一版，并保留灰度开关。
李雷负责整理发布清单，优先级高；韩梅梅负责通知客服团队，优先级中。
"""
response = model_with_structured_output.invoke(text)
rprint(response)

### 3.3 注意事项

- 不建议把标准库 `dataclass` 直接传给 `model.with_structured_output()`，不同 LangChain 版本支持不一致。
- 推荐路径是 `TypeAdapter(MyDataclass).json_schema()` 生成 schema，再用 `TypeAdapter.validate_python()` 把模型返回的 `dict` 转回 dataclass 实例。
- 字段描述写在 `field(metadata={"description": "..."})` 中，生成 JSON Schema 时会保留下来。
- `dataclass` 适合已有业务对象复用；如果你从零设计结构化输出 schema，Pydantic 通常更直接。

## 4. JSON Schema

JSON Schema 是最通用的结构化输出格式。Pydantic、TypedDict、dataclass 在很多框架内部最终也会被转换成 JSON Schema 或工具参数 schema。它适合跨语言、跨系统传递结构约束。

### 4.1 单对象：required + enum + 数值范围

In [ ]:
movie_schema = {
    "title": "MovieInfo",
    "description": "电影相关信息",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "电影名称",
        },
        "director": {
            "type": "string",
            "description": "导演",
        },
        "year": {
            "type": "integer",
            "description": "上映年份",
        },
        "genre": {
            "type": "string",
            "enum": ["剧情", "动作", "喜剧", "科幻", "其他"],
            "description": "电影类型，只能从枚举值中选择",
        },
        "rating": {
            "type": "number",
            "minimum": 0,
            "maximum": 10,
            "description": "评分，0-10 分",
        },
    },
    "required": ["title", "director", "year", "genre", "rating"],
    "additionalProperties": False,
}

model_with_structured_output = model.with_structured_output(movie_schema)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的主要信息")
rprint(structured_output)  # {'title': '肖申克的救赎', 'director': '弗兰克·德拉邦特', 'year': 1994, 'genre': '剧情', 'rating': 9.3}

### 4.2 数组与嵌套对象

In [ ]:
movie_with_cast_schema = {
    "title": "MovieWithCast",
    "description": "包含主演列表的电影信息",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影名称"},
        "year": {"type": "integer", "description": "上映年份"},
        "cast": {
            "type": "array",
            "description": "主要演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "饰演角色"},
                },
                "required": ["name", "role"],
            },
        },
    },
    "required": ["title", "year", "cast"],
}

model_with_structured_output = model.with_structured_output(movie_with_cast_schema)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的片名、年份和主要演员")
rprint(structured_output)

### 4.3 注意事项

- JSON Schema 可读性不如 Pydantic，复杂 schema 手写成本较高。
- 返回结果通常是普通 `dict`，不是 Pydantic 对象。
- `json_schema`、`function_calling`、`json_mode` 是不同结构化输出方法；当前 DeepSeek 示例更适合走默认的 `function_calling`，不要强行使用不支持的 `method="json_schema"`。
- 如果 schema 来自前端、接口文档或其他语言服务，JSON Schema 是最方便的共享格式。

## 5. 四种 schema 写法对比

### 5.1 核心区别

| 类型 | 返回结果 | 运行时校验 | 字段描述 | 适合场景 |
|------|----------|:---:|:---:|----------|
| Pydantic | Pydantic 对象 | ✅ 强 | ✅ `Field(...)` | 生产级抽取、字段校验、复杂嵌套结构 |
| TypedDict | `dict` | ❌ 弱 | ✅ `Annotated[...]` | 轻量结构、临时抽取、只需要字典结果 |
| dataclass | 先返回 `dict`，再用 `TypeAdapter` 转实例 | ⚠️ 中 | ✅ `field(metadata=...)` | 复用已有业务对象、轻量数据容器 |
| JSON Schema | `dict` | ⚠️ 取决于框架/额外校验器 | ✅ `description` | 跨语言共享 schema、前后端/接口统一约束 |

选择时不要只看“能不能传给模型”，还要看返回结果是否方便后续业务使用。

### 5.2 推荐使用场景

| 需求 | 推荐 |
|------|------|
| 希望返回对象，并自动校验字段 | Pydantic |
| 只需要简单 `dict`，不想引入 Pydantic 模型 | TypedDict |
| 项目里已有 dataclass 业务对象 | dataclass + `TypeAdapter` 生成 schema 并校验返回值 |
| schema 要给 JavaScript、Java、接口文档或其他系统复用 | JSON Schema |
| 字段多、嵌套深、约束复杂 | Pydantic 优先，必要时拆成多步 |
| 只做一次性演示或低风险抽取 | TypedDict 或 JSON Schema 都可以 |

### 5.3 常见注意点

- 结构化输出不是数据库约束。模型仍可能输出缺字段、错类型或截断内容，生产代码要保留校验和重试。
- 字段名要稳定、直白，避免 `info`、`data`、`value` 这类含义模糊的名称。
- 字段描述要说明“应该抽什么”，不要写成业务背景长文。
- 复杂嵌套结构容易失败，优先减少字段数量，或拆成多次提取。
- `Pydantic` 适合做最终可信数据边界；`TypedDict` 适合轻量声明；`dataclass` 适合复用业务对象；`JSON Schema` 适合系统间共享。

## 6. thinking 模式与 tool_choice 模式

本节只说明一个问题：为什么 `deepseek-v4-pro` 使用 `with_structured_output()` 默认会报错，以及不需要保留 thinking 时应该怎么选。

### 6.1 三种结构化输出机制

`with_structured_output()` 底层常见有 3 种格式控制方式：

| method | 约束方式 | DeepSeek 支持 | 特点 |
|--------|----------|:---:|------|
| json_schema | `response_format={"type":"json_schema"}` 精确约束 schema | ❌ | OpenAI 专有能力 |
| function_calling | 把 Pydantic schema 转成工具参数，通过 `tool_calls` 输出 | ✅ | 约束最强，是默认方式 |
| json_mode | `response_format={"type":"json_object"}`，只要求输出合法 JSON | ✅ | 不能约束字段名、类型和必填项 |

这里的“工具调用”不是执行真实函数，而是借用工具调用协议承载结构化 JSON。LangChain 拿到 `tool_calls` 后，会把参数解析成 Pydantic 对象。

### 6.2 报错原因

`deepseek-v4-pro` 默认开启 thinking。thinking 和强制 `tool_choice` 在协议层面互斥：

| 模式 | 响应字段 | 含义 |
|------|----------|------|
| thinking 开启 | `message.reasoning_content` + `message.content` | 先生成推理内容，再生成最终回复 |
| `tool_choice` 指定 | `message.tool_calls` | 直接生成工具调用参数 |

`with_structured_output(method="function_calling")` 会设置 `tool_choice`，要求模型必须走 `tool_calls` 路径。因此在 thinking 开启时，API 会直接返回 400：

```text
Thinking mode does not support this tool_choice
```

这个错误发生在 API 协议校验阶段，请求不会进入模型推理。它不是 Pydantic 解析失败，也不是模型“不够聪明”。

### 6.3 不保留 thinking：关闭 thinking + function_calling

如果目标是直接得到 Pydantic 对象，且不需要保留推理模式，最简单的做法是关闭 thinking，再使用默认的 `function_calling`：

```python
model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    extra_body={"thinking": {"type": "disabled"}},
)

model_with_structured_output = model.with_structured_output(Movie)
structured_output = model_with_structured_output.invoke("给出《肖申克的救赎》的详细信息")
rprint(structured_output)
```

适用范围：简单 schema 比较稳定；复杂嵌套 schema 仍可能出现截断或字段缺失。

### 6.4 常见失败类型

| 场景 | 现象 | 根本原因 | 处理方式 |
|------|------|----------|----------|
| `deepseek-v4-pro` + thinking 开启 + `function_calling` | `BadRequestError: Thinking mode does not support this tool_choice` | thinking 与强制 `tool_choice` 协议冲突 | 关闭 thinking，或改用 `bind_tools()` |
| `deepseek-chat` + 复杂 schema + `function_calling` | Pydantic `ValidationError`，字段缺失或 JSON 截断 | 模型没有稳定生成完整工具参数 | 简化 schema，或改用 `bind_tools()` / Agent 多步提取 |
| 任意模型 + `json_mode` | 字段名自由发挥、类型跑偏、额外字段增多 | `json_mode` 只保证合法 JSON，不保证符合 schema | 只用于低风险简单场景 |

判断问题时先看错误发生在哪里：400 报错是协议限制；Pydantic 报错是模型输出没有匹配 schema。

### 6.5 本节结论

| 需求 | 推荐方案 |
|------|----------|
| 简单 schema，不需要 thinking | `with_structured_output()` + 关闭 thinking |
| 需要保留 thinking | 不要用强制 `tool_choice`，见第 7 章的 `bind_tools()` 方案 |
| 复杂嵌套 schema | 优先考虑 `bind_tools()`，必要时拆成 Agent 多步提取 |

核心原则：`function_calling` 约束最强，但它依赖 `tool_choice`；thinking 开启时不能强制 `tool_choice`。

## 7. 不关闭 thinking 的结构化输出方案

本章讨论另一个目标：保留 `deepseek-v4-pro` 的 thinking，同时尽量得到可解析的结构化数据。

### 7.1 先区分三类调用

| 方式 | 请求次数 | 是否强制 `tool_choice` | thinking 兼容 | 返回结果 |
|------|:---:|:---:|:---:|----------|
| `with_structured_output(method="function_calling")` | 1 | ✅ | ❌ | 直接返回 Pydantic 对象 |
| `bind_tools()` + `.invoke()` | 1 | ❌ | ✅ | 返回 `AIMessage`，从 `tool_calls` 手动取参数 |
| ReAct Agent | 多次 | 取决于每步模型调用 | ✅ | 工具会被真实执行，结果回传给模型 |

`bind_tools()` 和 `with_structured_output()` 的关键差异不是请求次数，而是是否强制 `tool_choice`。`bind_tools()` 只声明工具可用，模型可以自主选择调用，因此不会触发 thinking 与 `tool_choice` 的协议冲突。

### 7.2 可选方案

#### 方案 A：`json_mode`（不推荐）

`json_mode` 与 thinking 不冲突，但只能保证输出是合法 JSON，不能保证字段名、字段类型和必填项符合 Pydantic schema。

```python
model = init_chat_model(model="deepseek-v4-pro", model_provider="deepseek")

model_with_structured_output = model.with_structured_output(Movie, method="json_mode")
structured_output = model_with_structured_output.invoke(
    "请以JSON格式给出《肖申克的救赎》的详细信息"
)
```

常见问题：`year` 可能变成 `releaseYear`，`director` 可能变成 `director_name`，`rating` 可能从 `float` 变成嵌套对象。除非 schema 很简单且容错空间很大，否则不建议作为主方案。

#### 方案 B：`bind_tools()`（推荐）

`bind_tools()` 不设置强制 `tool_choice`，thinking 可以保持开启。模型调用工具后，从 `result.tool_calls[0]["args"]` 取出参数，再构造 Pydantic 对象。

```python
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """电影相关信息"""
    title: str = Field(..., description="片名")
    director: str = Field(..., description="导演")
    year: int = Field(..., description="上映年份")
    rating: float = Field(..., description="评分 1-10")
    story: str = Field(..., description="剧情简介")

@tool
def save_movie(title: str, director: str, year: int, rating: float, story: str) -> str:
    """保存电影信息到数据库"""
    return "ok"

llm = model.bind_tools([save_movie])
result = llm.invoke("请用 save_movie 工具保存《肖申克的救赎》的完整信息")

movie = Movie(**result.tool_calls[0]["args"])
rprint(movie)
```

如果担心模型跳过工具调用，可以在 prompt 中明确要求“请调用 `save_movie` 工具”。

#### 方案 C：纯 prompt 输出 JSON（不推荐）

```python
model.invoke('请只输出 JSON：{"title":"肖申克的救赎", ...}')
```

这种方式没有协议约束，也没有 schema 校验。它适合临时演示，不适合作为稳定的数据抽取方案。

### 7.3 方案对比

| 方案 | 可靠性 | schema 约束 | thinking 兼容 | 代价 |
|------|:---:|:---:|:---:|------|
| 关闭 thinking + `with_structured_output()` | ✅ 简单 schema 稳定 | ✅ | ❌ | 不能保留 thinking |
| thinking + `json_mode` | ❌ | ❌ | ✅ | 需要自行处理字段跑偏 |
| thinking + `bind_tools()` | ✅ | ✅ | ✅ | 需要手动提取 `args` 并构造对象 |
| thinking + 纯 prompt | ❌ | ❌ | ✅ | 格式不可控 |

保留 thinking 时，优先选择 `bind_tools()`。它保留了工具参数 schema，又避开了强制 `tool_choice`。

### 7.4 最终选型

| 场景 | 推荐方案 | 原因 |
|------|----------|------|
| 简单 schema，不需要 thinking | 关闭 thinking + `with_structured_output()` | 代码最短，直接返回 Pydantic 对象 |
| 简单 schema，需要 thinking | `bind_tools()` | thinking 兼容，schema 约束仍然存在 |
| 复杂嵌套 schema | `bind_tools()` 或 Agent 多步提取 | 单次强制结构化输出更容易截断；拆分任务更稳 |
| 对可靠性要求很高 | Agent 多步提取 + 每步校验/重试 | 每一步 schema 更小，失败后可局部重试 |

一句话总结：能关 thinking 就用 `with_structured_output()`；不能关 thinking 就用 `bind_tools()`；schema 太复杂就拆成多步。